# Environment Setup

In [1]:
import sys
import os

#  Mount Google Drive
try:
    from google.colab import drive
    print("[INFO] Mounting Google Drive...")
    drive.mount('/content/drive')
except ImportError:
    print("❌ Error: Not running in a Colab environment!")

# 2. Inject your specific repo path
project_root = '/content/drive/MyDrive/CARDD-Tech-Diagnostic-Pipeline'
if project_root not in sys.path:
    sys.path.append(project_root)
    print(f"✅ Cloud path injected: {project_root}")


# Force Jupyter to reload custom modules automatically
try:
    import IPython
    shell = IPython.get_ipython()
    if shell is not None:
        shell.run_line_magic('load_ext', 'autoreload')
        shell.run_line_magic('autoreload', '2')
        print("🔄 Autoreload extension activated successfully.")
except Exception as e:
    print(f"⚠️ Autoreload failed to initialize: {e}")
    print("Pre-compiled modules will require manual kernel restarts upon modifications.")

import torch
print(f"🔥 PyTorch Version: {torch.__version__}")
print(f"💻 GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🏎️ GPU Device: {torch.cuda.get_device_name(0)}")

[INFO] Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Cloud path injected: /content/drive/MyDrive/CARDD-Tech-Diagnostic-Pipeline
⚠️ Autoreload failed to initialize: No module named 'imp'
Pre-compiled modules will require manual kernel restarts upon modifications.
🔥 PyTorch Version: 2.10.0+cu128
💻 GPU Available: True
🏎️ GPU Device: Tesla T4


# Loader Instantiation

In [8]:
from src.ingestion.motor_loader import MotorDataLoader

# Point this to wherever you dropped the raw dataset
DATA_PATH = f"{project_root}/data/raw_motor_signals.csv" 

print("[INFO] Initializing MotorLoader...")
try:
    loader = MotorDataLoader(data_path=DATA_PATH ,
                             sensor_cols=['Va', 'Vb', 'Vc', 'Ia', 'Ib', 'Ic'])
    print("✅ Loader initialized successfully.")
except Exception as e:
    print(f"❌ Failed to initialize loader: {e}")

[INFO] Initializing MotorLoader...
✅ Loader initialized successfully.


# NumPy Array Extraction (For Isolation Forest / scikit-learn)

In [9]:
print("[INFO] Testing get_numpy()...")

loader.load_and_clean()
X_np, y_np = loader.get_numpy()

print(f"Feature Matrix (X) Shape: {X_np.shape} | Type: {type(X_np)}")
print(f"Label Array (y) Shape: {y_np.shape} | Type: {type(y_np)}")

# Verify there are no corrupted NaN values hiding in the data
import numpy as np
assert not np.isnan(X_np).any(), "❌ Error: NaN values detected in Numpy array!"
print("✅ NumPy arrays are clean and ready for Stage 1 models.")

[INFO] Testing get_numpy()...
Feature Matrix (X) Shape: (10000, 6) | Type: <class 'numpy.ndarray'>
Label Array (y) Shape: (10000,) | Type: <class 'numpy.ndarray'>
✅ NumPy arrays are clean and ready for Stage 1 models.


# Tensor Extraction (For OC-NN / PyTorch)

In [11]:
print("[INFO] Testing get_tensors()...")

X_tensor, y_tensor = loader.get_tensors()

print(f"Feature Tensor Shape: {X_tensor.shape} | Type: {X_tensor.dtype}")
print(f"Label Tensor Shape: {y_tensor.shape} | Type: {y_tensor.dtype}")
print(f"Current Device: {X_tensor.device}")

# If a GPU is available, manually push the tensor to it (this is what your training loop will do)
if torch.cuda.is_available():
    X_tensor = X_tensor.to('cuda')
    y_tensor = y_tensor.to('cuda')
    print(f"   -> X_tensor updated device: {X_tensor.device}")

# If you have a GPU, the tensor should automatically be on 'cuda:0', not 'cpu'
if torch.cuda.is_available():
    assert X_tensor.device.type == 'cuda', "❌ Error: Tensors are not on the GPU!"

print("✅ Tensors are formatted and device-assigned correctly.")

[INFO] Testing get_tensors()...
Feature Tensor Shape: torch.Size([10000, 6]) | Type: torch.float32
Label Tensor Shape: torch.Size([10000]) | Type: torch.int64
Current Device: cpu
   -> X_tensor updated device: cuda:0
✅ Tensors are formatted and device-assigned correctly.
